In [1]:
# Imports
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# PySpark session and data loading
spark = SparkSession.builder.appName('IngredientClustering').getOrCreate()
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])
df = spark.read.schema(ingredients_schema).parquet('../output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/13 19:34:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Explode ingredients and build ingredient-food mapping
from pyspark.sql.functions import explode, col, collect_list
df_exploded = df.select('fdc_id', 'description', explode('all_ingredients').alias('ingredient'))
ingredient_foods = df_exploded.groupBy('ingredient').agg(collect_list('description').alias('foods'))
ingredient_foods_pd = ingredient_foods.toPandas()

NameError: name 'collect_list' is not defined

In [ ]:
# Clean and deduplicate ingredient names
ingredient_foods_pd['ingredient_clean'] = ingredient_foods_pd['ingredient'].str.lower().str.strip()
ingredient_foods_pd = ingredient_foods_pd.drop_duplicates('ingredient_clean').reset_index(drop=True)

In [ ]:
# Embed ingredient names
model = SentenceTransformer('all-MiniLM-L6-v2')
ingredient_embeddings = model.encode(ingredient_foods_pd['ingredient_clean'].tolist(), show_progress_bar=True, convert_to_numpy=True)

In [ ]:
# Cluster ingredients (KMeans, choose n_clusters=15 for demo, adjust as needed)
n_clusters = 15
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(ingredient_embeddings)
ingredient_foods_pd['cluster'] = labels

In [ ]:
# Visualize clusters with PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(ingredient_embeddings)
plt.figure(figsize=(10,7))
sns.scatterplot(x=emb_2d[:,0], y=emb_2d[:,1], hue=labels, palette='tab20', legend='full', s=40)
plt.title('Ingredient Clusters (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Example ingredients and foods per cluster

In [ ]:
# Show top 5 ingredients and example foods for each cluster
for c in range(n_clusters):
    cluster_df = ingredient_foods_pd[ingredient_foods_pd['cluster'] == c]
    print(f'--- Cluster {c} ---')
    print('Ingredients:', ', '.join(cluster_df['ingredient_clean'].head(5)))
    foods_examples = sum(cluster_df['foods'].head(5).tolist(), [])
    print('Example foods:', ', '.join(foods_examples[:5]))
    print()

## Save clustered ingredients and foods to CSVs

In [ ]:
# Save each cluster's ingredients and foods to CSV
output_dir = '../output/ingredient_categories'
os.makedirs(output_dir, exist_ok=True)
for c in range(n_clusters):
    cluster_df = ingredient_foods_pd[ingredient_foods_pd['cluster'] == c]
    out_df = cluster_df[['ingredient_clean', 'foods']].explode('foods').rename(columns={'ingredient_clean': 'ingredient', 'foods': 'food'})
    out_df.to_csv(f'{output_dir}/cluster_{c}.csv', index=False)